In [0]:
%python
%pip install yfinance

In [0]:
%python
# dml/02_carga_stg_incremental_cotacoes_fiis.ipynb
# MAGIC %pip install yfinance pandas numpy # Garante a instalação das bibliotecas no compute Serverless

# %%
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime
from delta.tables import DeltaTable

# %%
# 1. Recupera os tickers ativos na tabela dimensional de Fundos Imobiliários
catalogo = "product_dev"
schema = "financas"
tabela_dim = "dim_fundo_imobiliario"

print(f"Buscando tickers ativos na tabela {catalogo}.{schema}.{tabela_dim}...")

# Lê a tabela de dimensão usando Spark e converte para Pandas para processamento na API
df_tickers = spark.sql(f"SELECT ticker FROM {catalogo}.{schema}.{tabela_dim}").toPandas()

# Extrai a lista de tickers únicos e adiciona o sufixo '.SA' exigido pelo Yahoo Finance
tickers_b3 = df_tickers["ticker"].unique().tolist()
tickers_yf = [f"{t}.SA" for f in tickers_b3 for t in [f.strip()]]

print(f"Total de FIIs encontrados para monitoramento diário: {len(tickers_yf)}")

# %%
# 2. Faz o download incremental das cotações recentes via Yahoo Finance API
# O download em lote com delimitador de espaço (' ') otimiza a requisição e blinda contra bloqueios de IP (rate limit)
if len(tickers_yf) == 0:
    print("Aviso: Nenhum ticker encontrado na dimensão de FIIs.")
else:
    print("Baixando dados incrementais (últimos 5 dias) via Yahoo Finance API...")
    
    # Captura uma janela de 5 dias para garantir resiliência caso ocorram falhas em finais de semana ou feriados
    dados_brutos = yf.download(
        tickers=" ".join(tickers_yf), 
        period="5d",          # Janela curta para otimização de banda, processamento e custos Cloud
        interval="1d", 
        group_by="ticker", 
        actions=True,         # Captura a coluna de proventos/dividendos declarados no período
        progress=True
    )

    print("Download incremental concluído com sucesso!")

# %%
# 3. Transforma a matriz tridimensional do yfinance em um DataFrame bidimensional limpo
lista_registros = []

for ticker_sa in tickers_yf:
    ticker_original = ticker_sa.replace(".SA", "")
    
    # Validação nativa no primeiro nível das colunas para verificar se o ticker retornou dados na API
    if ticker_sa in dados_brutos.columns.get_level_values(0):
        df_ticker = dados_brutos[ticker_sa]
        
        # Garante a existência da coluna de fechamento e remove registros nulos/fantasmas do pregão
        if "Close" in df_ticker.columns:
            df_ticker_limpo = df_ticker.dropna(subset=["Close"])
            
            for data_pregao, row in df_ticker_limpo.iterrows():
                # Tratamento defensivo de fallbacks: evita falhas por campos ausentes ou valores NaN
                adj_close_val = float(row["Adj Close"]) if "Adj Close" in row and not np.isnan(row["Adj Close"]) else float(row["Close"])
                open_val = float(row["Open"]) if "Open" in row and not np.isnan(row["Open"]) else None
                high_val = float(row["High"]) if "High" in row and not np.isnan(row["High"]) else None
                low_val = float(row["Low"]) if "Low" in row and not np.isnan(row["Low"]) else None
                volume_val = int(row["Volume"]) if "Volume" in row and not np.isnan(row["Volume"]) else 0
                div_val = float(row["Dividends"]) if "Dividends" in row and not np.isnan(row["Dividends"]) else 0.0
                
                lista_registros.append({
                    "ticker": ticker_original,
                    "data_pregao": data_pregao, # Mantém o tipo Timestamp do Pandas para perfeita inferência de DateType no Spark
                    "preco_abertura": open_val,
                    "preco_maximo": high_val,
                    "preco_minimo": low_val,
                    "preco_fechamento": float(row["Close"]),
                    "preco_fechamento_ajustado": adj_close_val,
                    "volume_negociado": volume_val,
                    "proventos_pagos": div_val,
                    "data_carga": datetime.now()
                })

# Consolida a lista de dicionários em um DataFrame estruturado do Pandas
df_incremental_pandas = pd.DataFrame(lista_registros)

# %%
# 4. Converte os dados sanitizados para o ecossistema Spark
print(f"Processando {len(df_incremental_pandas)} linhas de cotações para inserção...")

# Converte o Pandas DataFrame para Spark DataFrame de forma nativa e segura
spark_df_incremental = spark.createDataFrame(df_incremental_pandas)

# Cria ou substitui a view temporária na memória do cluster Spark
spark_df_incremental.createOrReplaceTempView("temp_incremental_cotacoes")

# %%
# 5. Execução do MERGE INTO (Upsert) idempotente na tabela de Staging
tabela_destino = "stg_historico_cotacoes_fiis"
caminho_tabela_completo = f"{catalogo}.{schema}.{tabela_destino}"

print(f"Iniciando operação de MERGE INTO na tabela Delta: {caminho_tabela_completo}...")

# Instancia a tabela Delta de destino para aplicar a manipulação programática
tabela_delta = DeltaTable.forName(spark, caminho_tabela_completo)

# Executa o casamento de chaves por Ticker + Data do Pregão.
# Se o registro do dia já existir, atualiza os valores (ajustes de fechamento/dividendos). Se for inédito, insere.
tabela_delta.alias("target") \
    .merge(
        source=spark_df_incremental.alias("source"),
        condition="target.ticker = source.ticker AND target.data_pregao = source.data_pregao"
    ) \
    .whenMatchedUpdate(set={
        "preco_abertura": "source.preco_abertura",
        "preco_maximo": "source.preco_maximo",
        "preco_minimo": "source.preco_minimo",
        "preco_fechamento": "source.preco_fechamento",
        "preco_fechamento_ajustado": "source.preco_fechamento_ajustado",
        "volume_negociado": "source.volume_negociado",
        "proventos_pagos": "source.proventos_pagos",
        "data_carga": "source.data_carga"
    }) \
    .whenNotMatchedInsertAll() \
    .execute()

print(f"✅ Carga incremental diária via MERGE INTO concluída com SUCESSO em {caminho_tabela_completo}!")